Script for building base data catalogue from BIDS-compliant filenames.

Searches recursively within the provided MRI and fMRI data directories for paired .nii + .json files, and compiles metadata about all available neuroimaging data sessions for all subjects.

**NOTE:** Requires a variety of parameters to be set in 'config.yaml' file by user!

**Main output:**
- 'master_data_catalogue.csv' --> Lists out all subject_IDs and their available MRI & fMRI data, one row per subject
- This is the main data object used by subsequent scripts downstream, e.g. for flexible subsetting of subjects by session types, time-points, etc.
- Base table will (optionally) be further "decorated" by QC-related data from study documentation, if set up by user (may not be available for all datasets)

**Secondary output:**
- 'participants.tsv' --> a BIDS-compliant list of subject_IDs; not used in this pipeline but can sometimes be expected for other data-analysis packages

--------

In [ ]:
import yaml, os, json
from pathlib import Path
from bids import BIDSLayout
import pandas as pd
import numpy as np


# READ config.yaml for path- and parameter-setting:
CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)
    print(f"Loaded parameters from config.yaml:\n")

MRI_root = config['MRI_data_directory']
fMRI_root = config['fMRI_data_directory']

base_output_directory = config['root_output_directory']

HARD_STOP = config['hard_errors']

# Display loaded paths and parameters from config file:
print(f"--> Cataloging MRI data from path: {MRI_root}")
print(f"--> Cataloging fMRI data from path: {fMRI_root}")

if HARD_STOP == True:
    print("\n--> Error mode: 'STRICT' (hard errors will be thrown if unexpected/incompatible data are detected)")
elif HARD_STOP == False:
    print("\n--> Error mode: 'SOFT' (only printed warnings will be thrown, and subject_IDs with unexpected data will be dropped from data catalogue)")
else:
    print("\nUnexpected error mode parameter setting: proceeding in 'STRICT' mode")
    HARD_STOP = True

print(f"\nOutput data catalogue will be stored in: {base_output_directory}")

Build full base data catalog, by searching (recursively) through each of the specified MRI and fMRI input data directories (as per config.yaml): 

In [ ]:
# =========================
# Catalog (MRI + fMRI) into a single dataframe
# =========================

# Safety checks on root dirs:
for label, root in [("MRI_root", MRI_root), ("fMRI_root", fMRI_root)]:
    if not Path(root).exists():
        raise FileNotFoundError(f"{label} does not exist: {root}")

# Build BIDS layouts (tolerant; we only harvest entities if present):
layouts = []
for root in [MRI_root, fMRI_root]:
    try:
        layouts.append(BIDSLayout(root, validate=False))
        print(f"[info] Indexed: {root}")
    except Exception as e:
        print(f"[warn] Could not index {root}: {e}")
        layouts.append(None)

# Gather ONLY .nii/.nii.gz/.json BIDS files from both layouts (no filtering on suffix/datatype here):
rows = []
for lay in layouts:
    if lay is None:
        continue
    files = lay.get(return_type="object")  # <-- should grab any & all known 'BIDSFile' objects
    for f in files:
        p = Path(f.path)
        # robust filetype extension handling (e.g. supports '*.nii.gz' variants):
        ext = "".join(p.suffixes) if "".join(p.suffixes) in (".nii.gz",) else p.suffix
        if ext not in {".nii", ".nii.gz", ".json"}:
            continue

        # Entities (fill missing with np.nan):
        e = getattr(f, "entities", {}) or {}
        rows.append({
            "path":          str(p),
            "BIDS_extension": ext,
            "BIDS_datatype":  e.get("datatype", np.nan),
            "BIDS_suffix":    e.get("suffix", np.nan),
            "BIDS_subject":   e.get("subject", np.nan),
            "BIDS_session":   e.get("session", np.nan),
            "BIDS_task":      e.get("task", np.nan),
            "BIDS_run":       e.get("run", np.nan),
            "BIDS_acq":       e.get("acq", np.nan),
            "BIDS_dir":       e.get("dir", np.nan),
            "BIDS_echo":      e.get("echo", np.nan),
            "BIDS_space":     e.get("space", np.nan),
            "BIDS_desc":      e.get("desc", np.nan)})

df = pd.DataFrame(rows)

if df.empty:
    print("[info] No .nii/.nii.gz/.json files found under the provided roots.")
else:
    # Sort into a stable, human-friendly order (if columns exist):
    sort_cols = [c for c in ["BIDS_datatype","BIDS_suffix","BIDS_subject","BIDS_session","BIDS_task","BIDS_run","BIDS_echo","BIDS_acq","BIDS_dir","BIDS_space","BIDS_desc","BIDS_extension"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, kind="stable").reset_index(drop=True)

    # ========== Pairing logic: keep ONLY matched NIfTI+JSON base-name pairs ==========
    # Build pair_key = file path without final extension (i.e. strips '.nii' or '.nii.gz'):
    df["pair_key"] = df["path"].apply(lambda p: str(Path(p).with_suffix("")).replace(".nii", ""))

    # Group & classify:
    pairs, unpaired_nii, unpaired_json = [], [], []
    for key, g in df.groupby("pair_key", sort=False):
        exts = set(g["BIDS_extension"].tolist())
        has_nifti = bool({".nii", ".nii.gz"}.intersection(exts))
        has_json  = ".json" in exts
        if has_nifti and has_json:
            pairs.append(key)
        elif has_json and not has_nifti:
            unpaired_json.append(key)
        elif has_nifti and not has_json:
            unpaired_nii.append(key)

    # Report mismatches:
    if unpaired_nii or unpaired_json:
        print("[warn] Unpaired files detected:")
        for k in unpaired_nii:
            print(f"   - NIfTI without JSON: {k}")
        for k in unpaired_json:
            print(f"   - JSON without NIfTI: {k}")
    else:
        print("[info] All .nii/.nii.gz files have matching .json sidecars (and vice versa).")

    # Keep ONLY the NIfTI rows for matched pairs (drop JSON rows now, since they're redundant):
    keep_mask = df.apply(lambda r: (r["pair_key"] in pairs) and (r["BIDS_extension"] in {".nii", ".nii.gz"}), axis=1)
    df = df[keep_mask].reset_index(drop=True)
    print(f"[result] Kept {len(df)} matched NIfTI rows (dropped JSON sidecars and any unpaired files).")

    # ========== Minimal participants.tsv export (for downstream BIDS compliance) ==========
    # Build 'participant_id' from 'BIDS_subject' (add 'sub-' if missing); one row per subject:
    if "BIDS_subject" in df.columns and df["BIDS_subject"].notna().any():
        subj_series = df["BIDS_subject"].dropna().astype(str)
        subj_series = subj_series.apply(lambda s: s if s.startswith("sub-") else f"sub-{s}")
        participants_df = (
            df.dropna(subset=["BIDS_subject", "BIDS_suffix", "BIDS_session"])
            .assign(
                participant_id=lambda x: x["BIDS_subject"].apply(
                    lambda s: s if s.startswith("sub-") else f"sub-{s}"))
            .groupby(["participant_id", "BIDS_suffix"], as_index=False)
            .agg({"BIDS_session": lambda s: ",".join(sorted(set(s.astype(str))))})
            .pivot(index="participant_id", columns="BIDS_suffix", values="BIDS_session")
            .reset_index()
            .rename_axis(None, axis=1))
        participants_df = participants_df.rename(columns={c: f"{c}_sessions" for c in participants_df.columns if c != "participant_id"})
        participants_df = participants_df.fillna("n/a")
        out_path_participants = Path(base_output_directory) / "participants.tsv"
        participants_df.to_csv(out_path_participants, sep="\t", index=False, na_rep="n/a")
        print(f"[write] participants.tsv → {out_path_participants}  ({len(participants_df)} rows)")
    else:
        print("[info] No BIDS_subject values found; skipping participants.tsv export.")

    # ========== Column pruning & diagnostics ==========
    if df.empty:
        print("[info] df is empty after pairing; skipping column pruning.")
    else:
        # Drop any fully-empty columns:
        all_null_cols = [c for c in df.columns if df[c].isna().all()]
        if all_null_cols:
            print("[drop: no data] Columns dropped because they contain only NA/empty values:")
            for c in all_null_cols:
                print(f"   - {c}")
            df = df.drop(columns=all_null_cols)
        else:
            print("[drop: no data] None")

        # Report & drop single-valued (uninformative) columns (except for explicit exclusions):
        exclusion_list = ['BIDS_suffix', 'BIDS_subject', 'BIDS_session', 'BIDS_task']
        always_keep = {'path', 'pair_key'}  # <-- never drop

        single_val_info = []
        for c in df.columns:
            if c in always_keep:
                continue
            # Build a hashable view of the non-NA values:
            ser = df[c].dropna()
            # Convert potentially unhashable items inline:
            tmp_vals = []
            for x in ser:
                if isinstance(x, (list, tuple, np.ndarray)):
                    tmp_vals.append(tuple(x))
                elif isinstance(x, dict):
                    tmp_vals.append(tuple(sorted(x.items())))
                else:
                    try:
                        hash(x)
                        tmp_vals.append(x)
                    except TypeError:
                        tmp_vals.append(repr(x))
            unique_vals = pd.unique(pd.Series(tmp_vals))
            if len(unique_vals) == 1 and c not in exclusion_list:
                single_val_info.append((c, unique_vals[0]))

        if single_val_info:
            print("[drop: single-valued] Columns dropped because they had a single value across all rows:")
            for cname, val in single_val_info:
                print(f"   - {cname}: {val!r}")
            df = df.drop(columns=[cname for cname, _ in single_val_info])
        else:
            print("[drop: single-valued] None")

        # Print final summary:
        print(f"[result] df shape: {df.shape[0]} rows × {df.shape[1]} cols")
        with pd.option_context("display.max_rows", 5, "display.max_colwidth", 120):
            display(df.head())

Parse 'BIDS_session' column, and create a re-mapped column ('session_ID') based on mappings provided in the config.yaml file:

In [ ]:
# =========================
# Session ID remapping (config-driven)
# =========================

# Build reverse lookup: code -> human-readable label (e.g., "02SE01MR" -> "week2"):
session_ID_mappings = config.get("session_ID_mappings", {})
code_to_label = {}

# Detect conflicting mappings early:
for label, codes in (session_ID_mappings or {}).items():
    if codes is None:
        continue
    for code in codes:
        key = str(code).strip()
        if key in code_to_label and code_to_label[key] != label:
            raise ValueError(f"Conflicting mapping for code '{key}': '{code_to_label[key]}' vs '{label}'")
        code_to_label[key] = label

if "BIDS_session" not in df.columns:
    raise KeyError("Column 'BIDS_session' not found in df.")

# Check for unmapped session codes (ignore NaNs):
observed_codes = pd.Series(df["BIDS_session"]).dropna().astype(str).str.strip().unique()
unknown_codes = sorted([c for c in observed_codes if c not in code_to_label])

if unknown_codes:
    msg = "[session_ID mapping] Unmapped BIDS_session code(s) encountered: " + ", ".join(unknown_codes)
    if HARD_STOP:
        # Hard stop mode:
        raise ValueError(msg + ". Please add these under 'session_ID_mappings' in config.yaml.")
    else:
        # Soft mode: warn & drop all affected subjects/files:
        print("[warn]", msg)
        offending_subjects = df.loc[df["BIDS_session"].astype(str).isin(unknown_codes), "BIDS_subject"].dropna().unique()
        print(f"[warn] Dropping all rows for subjects with unmapped session codes: {', '.join(offending_subjects)}")
        df = df[~df["BIDS_subject"].isin(offending_subjects)].reset_index(drop=True)
else:
    print("[info] All BIDS_session values successfully mapped:")

# Create the new 'session_ID' column (mapped human-readable labels):
df["session_ID"] = df["BIDS_session"].apply(
    lambda x: code_to_label.get(str(x).strip()) if pd.notna(x) else np.nan)

# Print concise counts per session_ID (respecting the order in config):
counts = df["session_ID"].value_counts(dropna=True)
for label in session_ID_mappings.keys():
    print(f"\t{label} sessions:  \t{int(counts.get(label, 0))}")

# Optional: also report rows without a session_ID (i.e. if original 'BIDS_session' was NaN):
n_missing = int(df["session_ID"].isna().sum())
if n_missing:
    print(f"[info] rows without session_ID (NaN): {n_missing}")

Next step: Label files by data type (MRI or fMRI) via a re-mapping of the 'BIDS_suffix' column:

In [ ]:
# =========================
# Data type mapping (config-driven): BIDS_suffix -> data_type (e.g., MRI / fMRI)
# =========================

# Load mapping (destination keys = data_type labels; values = lists of BIDS_suffix codes):
datatype_mappings = config.get("datatype_mappings", {})
print("[info] datatype_mappings:", datatype_mappings)

if "BIDS_suffix" not in df.columns:
    raise KeyError("Column 'BIDS_suffix' not found in df.")

# Build reverse lookup: suffix -> data_type (e.g., "T1w" -> "MRI", "bold" -> "fMRI"), checking for conflicts:
suffix_to_type = {}
for dtype_label, suffixes in (datatype_mappings or {}).items():
    if suffixes is None:
        continue
    for suf in suffixes:
        key = str(suf).strip()
        if key in suffix_to_type and suffix_to_type[key] != dtype_label:
            raise ValueError(f"Conflicting mapping for suffix '{key}': '{suffix_to_type[key]}' vs '{dtype_label}'")
        suffix_to_type[key] = dtype_label

# Confirm that all observed suffixes are mappable (ignore NaNs):
observed_suffixes = pd.Series(df["BIDS_suffix"]).dropna().astype(str).str.strip().unique()
unknown_suffixes = sorted([s for s in observed_suffixes if s not in suffix_to_type])

if unknown_suffixes:
    msg = "[data_type mapping] Unmapped BIDS_suffix value(s) encountered: " + ", ".join(unknown_suffixes)
    if HARD_STOP:
        # Hard stop mode:
        raise ValueError(msg + ". Please add these under 'datatype_mappings' in config.yaml.")
    else:
        # Soft mode: warn & drop only offending rows (not whole subjects):
        print("[warn]", msg)
        print("[warn] Dropping rows with these unmapped suffixes (rows only, not entire subjects).")
        df = df[~df["BIDS_suffix"].astype(str).isin(unknown_suffixes)].reset_index(drop=True)
else:
    print("[info] All BIDS_suffix values successfully mapped.")

# Create the new 'data_type' column:
df["data_type"] = df["BIDS_suffix"].apply(
    lambda x: suffix_to_type.get(str(x).strip()) if pd.notna(x) else np.nan)

# Print concise counts per data_type (respect order as in config):
counts = df["data_type"].value_counts(dropna=True)
for dtype_label in datatype_mappings.keys():
    print(f"\t{dtype_label} files found: {int(counts.get(dtype_label, 0))}")

# Optional: report rows without a data_type (i.e. if BIDS_suffix was NaN):
n_missing_dtype = int(df["data_type"].isna().sum())
if n_missing_dtype:
    print(f"[info] rows without data_type (NaN): {n_missing_dtype}")

Next, we assign group_IDs based on substrings found in the original filepaths; if no strings are given (or matches are not found), group_ID gets assigned 'UNKNOWN' by default.

In [ ]:
# =========================
# Group assignment from path substrings (config-driven)
# =========================

group_identifiers = config.get("group_identifiers", {})
if "path" not in df.columns:
    raise KeyError("Column 'path' not found in df.")
if "BIDS_subject" not in df.columns:
    raise KeyError("Column 'BIDS_subject' not found in df.")

# Create 'group_ID' for all rows:
if not group_identifiers:  # <-- e.g. if empty dict or missing
    print("[warn] No 'group_identifiers' found in config.yaml; assigning group_ID='UNKNOWN' for all rows.")
    df["group_ID"] = "UNKNOWN"
else:
    # Build a normalized, case-insensitive search over path:
    df["_path_lc"] = df["path"].astype(str).str.lower()

    # Prepare storage:
    assigned = []        # <-- single chosen group key per row (or None)
    matched_multi = []   # <-- number of matched groups per row (for diagnostics)
    matched_groups_col = []  # <-- list of matched group keys per row

    # Iterate rows and find matching group keys by substring (case-insensitive):
    for i in range(len(df)):
        p = df["_path_lc"].iloc[i]
        hits = []
        # loop over mapping keys and their substrings:
        for gkey, substrings in group_identifiers.items():
            if substrings is None:
                continue
            for sub in substrings:
                if sub is None:
                    continue
                if str(sub).lower() in p:
                    hits.append(gkey)
                    break  # <-- one hit per group key is enough

        hits_unique = sorted(set(hits))
        matched_groups_col.append(hits_unique)
        matched_multi.append(len(hits_unique))
        if len(hits_unique) == 1:
            assigned.append(hits_unique[0])
        else:
            assigned.append(None)  # <-- None for 0 or >1 hits; case handled below

    df["group_ID"] = pd.Series(assigned, index=df.index).fillna("UNKNOWN")

    # Conflict detection at subject level (e.g. any subject mapped to >1 distinct non-UNKNOWN groups):
    subj_groups = {}
    for i in range(len(df)):
        subj = df["BIDS_subject"].iloc[i]
        gid  = df["group_ID"].iloc[i]
        if pd.isna(subj):
            continue
        if gid == "UNKNOWN":
            continue
        subj_groups.setdefault(subj, set()).add(gid)

    conflicting_subjects = sorted([s for s, gs in subj_groups.items() if len(gs) > 1])

    # Also flag rows where a single path matched multiple groups (extra sanity-check):
    row_conflict_subjects = df.loc[pd.Series(matched_multi, index=df.index) > 1, "BIDS_subject"].dropna().unique().tolist()
    for s in row_conflict_subjects:
        if s not in conflicting_subjects:
            conflicting_subjects.append(s)
    conflicting_subjects = sorted(conflicting_subjects)

    if conflicting_subjects:
        msg = "[group_ID] Conflicting group matches for subjects: " + ", ".join(map(str, conflicting_subjects))
        if HARD_STOP:
            raise ValueError(msg + ". Resolve overlapping substrings in 'group_identifiers' or adjust paths.")
        else:
            print("[warn]", msg)
            print("[warn] Dropping all rows for these subjects due to ambiguous group assignment.")
            df = df[~df["BIDS_subject"].isin(conflicting_subjects)].reset_index(drop=True)

    # Clean up temp column:
    if "_path_lc" in df.columns:
        df = df.drop(columns=["_path_lc"])

# Display summary counts per configured group key:
counts = df["group_ID"].value_counts(dropna=False)
print("Number of 'group_ID' column assignments (number of files; not unique subject_IDs):")
for gkey in group_identifiers.keys():
    print(f"\t{gkey}: {int(counts.get(gkey, 0))}")

# Optional: report UNKNOWN count, if any:
unknown_n = int(counts.get("UNKNOWN", 0))
if unknown_n:
    print(f"Number of UNKNOWN group_ID assignments: {unknown_n}")

Next step: remove redunant substrings from BIDS_subject identifiers, and place streamlined identifiers into 'subject_ID' column:

In [ ]:
# =========================
# Build subject_ID from BIDS_subject (config-driven substring stripping)
# =========================

if "BIDS_subject" not in df.columns:
    raise KeyError("Column 'BIDS_subject' not found in df.")

strip_list = config.get("strip_subjectID_substrings", None)

# If not provided or empty -> no changes:
if not strip_list:
    print("No redundant subject_ID substrings set in CONFIG; using full BIDS-derived subject identifiers.")
    df["subject_ID"] = df["BIDS_subject"]
else:
    # Make a working copy as strings (preserve original column):
    subj_series = df["BIDS_subject"].astype(str)
    # Track counts of affected identifiers per substring (count unique IDs touched):
    removal_counts = {}

    for sub in strip_list:
        if sub is None:
            continue
        sub = str(sub)  # <-- ensure string
        # Identify which identifiers contain this exact (case-sensitive) substring:
        has_sub = subj_series.str.contains(sub, na=False)
        # Count unique BIDS_subject identifiers affected (not occurrences):
        affected_ids = df.loc[has_sub, "BIDS_subject"].dropna().astype(str).unique()
        removal_counts[sub] = len(affected_ids)
        # Perform the replacement (remove all occurrences of the substring):
        subj_series = subj_series.str.replace(sub, "", regex=False)

    # Assign the result:
    df["subject_ID"] = subj_series

    # Print concise summary for substrings that actually triggered removals:
    any_removed = False
    for substring, count in removal_counts.items():
        if count == 0:
            print(f"No instances of substring: '{substring}' encountered in 'BIDS_subject' identifiers; check CONFIG value provided (Note: substrings are CASE-SENSITIVE).")
        if count > 0:
            any_removed = True
            print(f"Removed the substring: '{substring}' from {count} 'BIDS_subject' identifiers.")
    if not any_removed:
        print("No subject_ID substrings matched; 'subject_ID' remains identical to 'BIDS_subject'.")

Next step: Pivot full table to one row per subject, create & fill synthetic data cataloguing columns (e.g. 'has_MRI', etc.), as well as set placeholder columns for QC decoration:

In [ ]:
# =========================
# Pivot table to one row per subject_ID with per-session/per-type columns
# =========================

# Required columns' sanity-checks:
required_columns = ["subject_ID", "group_ID", "data_type", "session_ID", "pair_key"]
missing_required_columns = [column for column in required_columns if column not in df.columns]
if missing_required_columns:
    raise KeyError(f"Missing required columns in df: {missing_required_columns}")

# Optional: refine 'session_ID' using 'BIDS_run' when multiple runs exist:
if "BIDS_run" in df.columns:
    run_subset = df.dropna(subset=["data_type", "session_ID", "BIDS_run"]).copy()
    run_subset["data_type"] = run_subset["data_type"].astype(str)
    run_subset["session_ID"] = run_subset["session_ID"].astype(str)
    run_subset["BIDS_run"] = run_subset["BIDS_run"].astype(str).str.strip()

    run_labels_by_type_and_session = {}
    if not run_subset.empty:
        for (current_data_type, current_session_id), group in run_subset.groupby(
                ["data_type", "session_ID"]):
            run_labels = (
                group["BIDS_run"]
                .dropna()
                .astype(str)
                .str.strip()
                .unique()
                .tolist())
            if len(run_labels) > 1:
                try:
                    run_labels_sorted = sorted(run_labels, key=lambda value: int(value))
                except ValueError:
                    run_labels_sorted = sorted(run_labels)
                run_labels_by_type_and_session[(current_data_type, current_session_id)] = run_labels_sorted

    if run_labels_by_type_and_session:
        new_session_id_values = []
        for index, row in df.iterrows():
            original_session_id = row["session_ID"]
            if pd.isna(original_session_id) or pd.isna(row.get("data_type", np.nan)):
                new_session_id_values.append(original_session_id)
                continue

            bids_run_value = row.get("BIDS_run", np.nan)
            if pd.isna(bids_run_value):
                new_session_id_values.append(original_session_id)
                continue

            current_data_type = str(row["data_type"])
            current_session_id = str(original_session_id)
            key = (current_data_type, current_session_id)

            run_label_list = run_labels_by_type_and_session.get(key, None)
            if run_label_list is None:
                new_session_id_values.append(original_session_id)
                continue

            run_label_string = str(bids_run_value).strip()
            if run_label_string not in run_label_list:
                new_session_id_values.append(original_session_id)
                continue

            run_index = run_label_list.index(run_label_string) + 1
            new_session_id_values.append(f"{original_session_id}-{run_index}")

        df["session_ID"] = pd.Series(new_session_id_values, index=df.index)

# Sanity check: each 'subject_ID' must map to a single 'group_ID':
unique_group_counts_per_subject = (
    df[["subject_ID", "group_ID"]]
      .dropna(subset=["subject_ID"])
      .groupby("subject_ID")["group_ID"]
      .nunique(dropna=True))

conflicting_group_subject_ids = unique_group_counts_per_subject[unique_group_counts_per_subject > 1].index.tolist()
if conflicting_group_subject_ids:
    conflict_message = "[group_ID] A subject_ID has multiple group_ID values: " + ", ".join(conflicting_group_subject_ids)
    if HARD_STOP:
        raise ValueError(conflict_message)
    else:
        print("[warn]", conflict_message)
        print("[warn] Dropping all rows for these conflicting subjects.")
        df = df[~df["subject_ID"].isin(conflicting_group_subject_ids)].reset_index(drop=True)

# Determine the unique data types present:
unique_data_types = (
    pd.Series(df["data_type"])
      .dropna()
      .astype(str)
      .sort_values()
      .unique()
      .tolist())

# Build per-type session lists (only session_IDs that actually occur for each data_type):
sessions_by_type = {}
for current_data_type in unique_data_types:
    subset = df[df["data_type"].astype(str) == current_data_type]
    session_values = (
        subset["session_ID"]
        .dropna()
        .astype(str)
        .sort_values()
        .unique()
        .tolist())
    sessions_by_type[current_data_type] = session_values

# --- Detect duplicates ---
# Base key: (subject_ID, data_type, session_ID):
group_columns_for_duplicates = ["subject_ID", "data_type", "session_ID"]

# Only include BIDS_run if it exists AND has more than one distinct non-NaN value:
if "BIDS_run" in df.columns:
    non_null_runs = df["BIDS_run"].dropna()
    if non_null_runs.nunique() > 1:
        group_columns_for_duplicates.append("BIDS_run")

duplicate_key_counts = (
    df.dropna(subset=["subject_ID", "data_type", "session_ID"])
      .groupby(group_columns_for_duplicates)
      .size()
      .reset_index(name="row_count"))

subject_ids_with_duplicates = (
    duplicate_key_counts.loc[duplicate_key_counts["row_count"] > 1, "subject_ID"]
    .drop_duplicates()
    .tolist())

if subject_ids_with_duplicates:
    duplicates_message = (
        f"[dup] Detected {len(subject_ids_with_duplicates)} subject(s) with >1 file for the same "
        f"key based on {group_columns_for_duplicates}.")
    if HARD_STOP:
        preview_subject_ids = ", ".join(subject_ids_with_duplicates[:10])
        raise ValueError(duplicates_message + f" Offenders include: {preview_subject_ids} ...")
    else:
        print("[warn]", duplicates_message)
        print("[warn] Example subjects (first 10):", ", ".join(subject_ids_with_duplicates[:10]))

# Prepare a quick lookup to flag subjects with any duplicates:
has_duplicate_lookup = {subject_id: False for subject_id in df["subject_ID"].dropna().unique()}
for subject_id in subject_ids_with_duplicates:
    has_duplicate_lookup[subject_id] = True

# Build the pivoted per-subject table:
unique_subject_ids_sorted = (
    pd.Series(df["subject_ID"])
      .dropna()
      .astype(str)
      .sort_values()
      .unique()
      .tolist())

pivoted_subject_rows = []
for subject_id in unique_subject_ids_sorted:
    subject_slice = df[df["subject_ID"] == subject_id]

    # Single group_ID per subject (ensured by earlier check / drop):
    subject_group_values = subject_slice["group_ID"].dropna().astype(str).unique().tolist()
    subject_group_id = subject_group_values[0] if subject_group_values else "UNKNOWN"

    # Boolean coverage flags:
    has_mri_flag = bool((subject_slice["data_type"].astype(str) == "MRI").any())
    has_fmri_flag = bool((subject_slice["data_type"].astype(str) == "fMRI").any())

    # Counts of files by type:
    num_mri_files = int((subject_slice["data_type"].astype(str) == "MRI").sum())
    num_fmri_files = int((subject_slice["data_type"].astype(str) == "fMRI").sum())

    # Initialize the base record:
    subject_record = {
        "subject_ID": subject_id,
        "group_ID": subject_group_id,
        "has_MRI": has_mri_flag,
        "has_fMRI": has_fmri_flag,
        "n_MRI": num_mri_files,
        "n_fMRI": num_fmri_files,
        "n_good_MRIs": np.nan,
        "n_good_fMRIs": np.nan,
        "first_good_MRI": np.nan,
        "first_good_fMRI": np.nan,
        "HAS_DUPLICATE": has_duplicate_lookup.get(subject_id, False)}

    # Dynamic filename/QC columns for each <'data_type' x 'session_ID'> that actually exists for that 'data_type':
    for current_data_type in unique_data_types:
        session_list = sessions_by_type.get(current_data_type, [])
        for current_session_id in session_list:
            matching_subset = subject_slice[
                (subject_slice["data_type"].astype(str) == current_data_type) &
                (subject_slice["session_ID"].astype(str) == current_session_id)]

            # Base filename from pair_key (strip directory), or np.nan if not present:
            if matching_subset.empty:
                base_filename = np.nan
            else:
                sorted_subset = matching_subset.sort_values("pair_key", kind="stable")
                base_filename = Path(sorted_subset["pair_key"].iloc[0]).name

            filename_column_name = f"{current_data_type}_{current_session_id}_filename"
            qc_summary_column_name = f"{current_data_type}_{current_session_id}_QC_summary"
            qc_comment_column_name = f"{current_data_type}_{current_session_id}_QC_comment"

            subject_record[filename_column_name] = base_filename
            subject_record[qc_summary_column_name] = np.nan
            subject_record[qc_comment_column_name] = np.nan

    pivoted_subject_rows.append(subject_record)

pivot_df = pd.DataFrame(pivoted_subject_rows)

# Final column ordering: "core" columns first, then dynamic filename/QC columns:
core_column_order = [
    "subject_ID",
    "group_ID",
    "has_MRI",
    "has_fMRI",
    "n_MRI",
    "n_fMRI",
    "n_good_MRIs",
    "n_good_fMRIs",
    "first_good_MRI",
    "first_good_fMRI",
    "HAS_DUPLICATE"]

dynamic_columns_in_order = []
for current_data_type in unique_data_types:
    session_list = sessions_by_type.get(current_data_type, [])
    for current_session_id in session_list:
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_filename")
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_QC_summary")
        dynamic_columns_in_order.append(f"{current_data_type}_{current_session_id}_QC_comment")

all_columns_now = pivot_df.columns.tolist()
final_column_order = [c for c in core_column_order if c in all_columns_now] \
                     + [c for c in dynamic_columns_in_order if c in all_columns_now]

pivot_df = pivot_df[final_column_order]

print(f"[result] pivoted_df shape: {pivot_df.shape[0]} rows × {pivot_df.shape[1]} cols")
duplicate_subject_total = int(pivot_df["HAS_DUPLICATE"].sum()) if "HAS_DUPLICATE" in pivot_df.columns else 0
if duplicate_subject_total:
    print(f"[warn] {duplicate_subject_total} subject(s) have duplicate files for at least one combination of {group_columns_for_duplicates}.")

In [ ]:
pivot_df.sample(7)

---------
#### Final save / export:

- We'll add additional QC decorations in another script, as these are optional and may not always be available in the same format as the current dataset.

In [ ]:
# Convert path string from config.yaml into valid Path object:
base_output_directory = Path(base_output_directory)

output_file_path = base_output_directory / "master_data_catalogue.csv"

# Export the dataframe to .csv:
pivot_df.to_csv(output_file_path, index=False)

print(f"[write] master_data_catalogue.csv → {output_file_path.resolve()}")
print(f"[result] Exported {pivot_df.shape[0]} rows × {pivot_df.shape[1]} columns.")